# Squat-to-Stand Verification — step by step

The notebook form of `reports/s2s_report.html`
(`uv run python -m boneid.report`). Same pipeline, same numbers,
one cell per step.

## How to run this notebook

```bash
uv run jupyter lab notebooks/s2s_report.ipynb          # interactive
uv run jupyter nbconvert --to notebook --execute --inplace notebooks/s2s_report.ipynb
```

The second form is what CI does: it runs every cell top to bottom and writes the
outputs back into the file, so a clean execution is the test.

## How to convert it to a script

```bash
uv run jupyter nbconvert --to script notebooks/s2s_report.ipynb
```

which writes `notebooks/s2s_report.py` — the same cells, minus the prose, runnable
with `uv run python notebooks/s2s_report.py`.

## What is *not* in here

No mathematics. Every number below comes from `boneid.core`, `boneid.io_v3d`
and `boneid.report`; the notebook only calls them, arranges the figures and
narrates. `uv run pytest tests/test_key.py` checks the same claims as
assertions.


## Setup

`simulate_squat_to_stand` prescribes the motion (minimum-jerk segment
tilts), welds the foot to the ground, and computes the ground reaction
wrench *exactly* from whole-body Newton–Euler — so nothing below is
being compared against another implementation, only against physics.
Filtering is off (`lowpass_hz=0`, `force_lowpass_hz=0`): the data is
analytic, so filtering it would only add error.

In [1]:
import numpy as np

from boneid.core import AnalysisParams, energy_audit, inverse_dynamics
from boneid.simulate import simulate_squat_to_stand

import matplotlib.pyplot as plt
from IPython.display import HTML, display

from boneid import report as R
from boneid.report import (JOINT_COLORS, FAINT, NEUTRAL, new_fig, new_grid,
                           style_axes)


def show(fig):
    """Render a matplotlib figure inline as an SVG, via the report's own
    `fig_svg` — identical output to the HTML report, no second code path.

    (`boneid.report` forces the Agg backend at import so report generation
    works headless; going through `fig_svg` sidesteps the backend entirely.)"""
    display(HTML(f'<img src="{R.fig_svg(fig)}" style="max-width:100%">'))

In [2]:
params = AnalysisParams(lowpass_hz=0.0, force_lowpass_hz=0.0)
skeleton, kin, ground, truth = simulate_squat_to_stand()
t = kin.t
joints = skeleton.joint_names
bw = skeleton.mass.sum() * 9.81
print(f"{len(skeleton.segment_names)} segments: "
      f"{', '.join(skeleton.segment_names)}")
print(f"joints: {', '.join(joints)}")
print(f"body mass {skeleton.mass.sum():.1f} kg  (body weight {bw:.0f} N)")
print(f"{len(t)} frames at {kin.rate:.0f} Hz = {t[-1]:.2f} s")

4 segments: foot, shank, thigh, torso
joints: ankle, knee, hip
body mass 64.8 kg  (body weight 636 N)
1000 frames at 500 Hz = 2.00 s


## Step 1 — the prescribed movement

Minimum-jerk tilts from a deep squat to standing. Velocity and
acceleration vanish at both endpoints, which is what gives us analytic
static torques to check against later.

In [3]:
fig, ax = new_fig(height=2.9)
for s_i, name in enumerate(skeleton.segment_names):
    ax.plot(t, np.degrees(truth["tilt"][:, s_i]), color=JOINT_COLORS[s_i],
            lw=1.8, label=name)
style_axes(ax, "time (s)", "segment tilt from vertical (deg)")
ax.legend(frameon=False, fontsize=9, ncol=4, loc="upper right")
show(fig)

## Step 2 — the ground reaction, computed not modelled

The simulator solves whole-body Newton–Euler for the wrench the floor
must apply. Unweighting, a propulsive overshoot above body weight, then
settling to standing — none of that was prescribed.

In [4]:
fig, ax = new_fig(height=2.9)
ax.plot(t, ground.force[:, 2], color=JOINT_COLORS[0], lw=1.8, label="vertical")
ax.plot(t, ground.force[:, 0], color=JOINT_COLORS[1], lw=1.8, label="fore-aft")
ax.axhline(bw, color=FAINT, lw=1.0, ls="--")
ax.annotate("body weight", (t[-1], bw), ha="right", va="bottom", fontsize=8.5,
            color=NEUTRAL)
style_axes(ax, "time (s)", "ground reaction force (N)")
ax.legend(frameon=False, fontsize=9)
show(fig)

## Step 3 — inverse dynamics

`core.inverse_dynamics` runs the bottom-up Newton–Euler recursion:
the ground wrench is the distal load on the foot, each segment's
proximal reaction becomes the next one's distal load. Inertia is
transformed with the full similarity `R I Rᵀ` and Euler's equation
keeps the gyroscopic term `ω×(Iω)` — the legacy MATLAB got both wrong,
which is exactly what the open circles below would expose.

In [5]:
idres = inverse_dynamics(skeleton, kin, ground, params)

fig, ax = new_fig(height=3.2)
for j, name in enumerate(joints):
    ax.plot(t, idres.joint_torque[:, j, 1], color=JOINT_COLORS[j], lw=1.8,
            label=name)
    for frame, key in ((0, "static_torque_start"), (-1, "static_torque_end")):
        ax.plot(t[frame], truth[key][name][1], "o", ms=7, mfc="none",
                mec=JOINT_COLORS[j], mew=1.6)
ax.plot([], [], "o", ms=7, mfc="none", mec=NEUTRAL, mew=1.6,
        label="analytic static value")
style_axes(ax, "time (s)", "sagittal joint torque (N m)")
ax.legend(frameon=False, fontsize=9, ncol=2)
show(fig)

for j, name in enumerate(joints):
    for frame, key in ((0, "static_torque_start"), (-1, "static_torque_end")):
        got = idres.joint_torque[frame, j, 1]
        want = truth[key][name][1]
        print(f"{name:6s} {key[14:]:5s}  ours {got:9.3f}  analytic {want:9.3f}"
              f"  diff {got - want:+.2e} N m")

ankle  start  ours    57.307  analytic    57.175  diff +1.32e-01 N m
ankle  end    ours     3.302  analytic     3.110  diff +1.93e-01 N m
knee   start  ours  -106.617  analytic  -106.527  diff -8.92e-02 N m
knee   end    ours    -4.403  analytic    -4.512  diff +1.10e-01 N m
hip    start  ours   116.197  analytic   116.047  diff +1.51e-01 N m
hip    end    ours     2.561  analytic     2.543  diff +1.83e-02 N m


## Step 4 — the residual wrench

The wrench that would have to act at the top of the torso for its
balance to close. On simulated data it must vanish to round-off; on
real data (the P1 notebook) it is large, structured and *meaningful*.
Same output, two completely different readings.

In [6]:
interior = slice(5, -5)
fig, axes = new_fig(2, height=2.7)
for c, lab in enumerate("xyz"):
    axes[0].plot(t, idres.residual_force[:, c], color=JOINT_COLORS[c], lw=1.4,
                 label=f"F{lab}")
    axes[1].plot(t, idres.residual_torque[:, c], color=JOINT_COLORS[c], lw=1.4,
                 label=f"M{lab}")
style_axes(axes[0], "time (s)", "residual force (N)")
axes[0].legend(frameon=False, fontsize=8.5)
style_axes(axes[1], "time (s)", "residual torque (N m)")
axes[1].legend(frameon=False, fontsize=8.5)
show(fig)

res_f = np.abs(idres.residual_force[interior]).max()
res_t = np.abs(idres.residual_torque[interior]).max()
print(f"peak residual force  {res_f:.2e} N     ({res_f / bw:.1e} BW)")
print(f"peak residual torque {res_t:.2e} N m")

peak residual force  2.27e-13 N     (3.6e-16 BW)
peak residual torque 2.13e-04 N m


## Step 5 — the energy audit

`core.energy_audit` compares d(KE+PE)/dt against the summed power of
every wrench in the model. Energy is never used inside the
inverse-dynamics recursion, so this is an independent instrument, not
a tautology.

In [7]:
audit = energy_audit(skeleton, kin, ground, idres, params)

fig, ax = new_fig(height=3.0)
pe0 = audit.potential[0]
ax.plot(t, audit.kinetic, color=JOINT_COLORS[1], lw=1.8, label="kinetic")
ax.plot(t, audit.potential - pe0, color=JOINT_COLORS[0], lw=1.8,
        label="potential (rel.)")
ax.plot(t, audit.kinetic + audit.potential - pe0, color=NEUTRAL, lw=2.2,
        label="total")
style_axes(ax, "time (s)", "energy (J)")
ax.legend(frameon=False, fontsize=9)
show(fig)

fig, axes = new_fig(2, height=2.9)
axes[0].plot(t, audit.de_dt, color=NEUTRAL, lw=2.6, alpha=0.45,
             label="d(KE+PE)/dt")
axes[0].plot(t, audit.power_total, color=JOINT_COLORS[0], lw=1.2,
             label="joint + ground + residual power")
style_axes(axes[0], "time (s)", "power (W)")
axes[0].legend(frameon=False, fontsize=8.5)
axes[1].plot(t[interior], audit.imbalance[interior], color=JOINT_COLORS[1],
             lw=1.4)
style_axes(axes[1], "time (s)", "imbalance (W)")
show(fig)

imb = np.abs(audit.imbalance[interior]).max()
scale = np.abs(audit.de_dt).max()
print(f"peak imbalance {imb:.3e} W  =  {imb / scale:.1e} of peak dE/dt "
      f"({scale:.1f} W)")

peak imbalance 7.861e-04 W  =  1.5e-06 of peak dE/dt (538.0 W)


## Step 6 — joint powers

Where the work is done. The hip and knee raise the large torso; the
ground wrench does no work on the welded foot and the residual power is
negligible, so the joint powers alone carry ΔE.

In [8]:
fig, ax = new_fig(height=3.0)
for j, name in enumerate(joints):
    ax.plot(t, audit.joint_power[:, j], color=JOINT_COLORS[j], lw=1.8,
            label=name)
ax.plot(t, audit.ground_power + audit.residual_power, color=FAINT, lw=1.2,
        label="ground + residual")
style_axes(ax, "time (s)", "power (W)")
ax.legend(frameon=False, fontsize=9)
show(fig)

work = np.trapezoid(audit.joint_power.sum(axis=1), t)
de = (audit.kinetic[-1] + audit.potential[-1]
      - audit.kinetic[0] - audit.potential[0])
print(f"integral of joint power  {work:.3f} J")
print(f"total energy change      {de:.3f} J   (difference {work - de:+.2e} J)")

integral of joint power  445.958 J
total energy change      445.958 J   (difference +1.30e-05 J)


## Step 7 — the movement in 3-D

`boneid.viz` drives a headless meshcat server and exports a
self-contained animated page. It is written next to the report rather
than embedded, so this notebook stays small.

In [9]:
from pathlib import Path

viewer_path = Path("../reports/s2s_viewer.html").resolve()
try:
    from boneid import viz
    vis = viz.start_viewer()
    viz.animate(vis, skeleton, kin, ground=ground, decimate=10,
                repetitions=10000)
    viz.save_html(vis, viewer_path)
    viz.stop_viewer(vis)
    print(f"wrote {viewer_path} "
          f"({viewer_path.stat().st_size / 1e6:.1f} MB)")
    display(HTML(f'<iframe src="{viewer_path.as_uri()}" width="100%" '
                 f'height="420" style="border:1px solid #ddd"></iframe>'))
except Exception as exc:
    print(f"viz skipped: {exc}")

wrote /Users/jeremy/Git/brent/bonelab_inverse_dynamics/reports/s2s_viewer.html (1.0 MB)


/Users/jeremy/Git/brent/bonelab_inverse_dynamics/.venv/lib/python3.12/site-packages/IPython/core/display.py:452: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


## Reproducing the HTML report

Everything above, plus the prose, is what
`uv run python -m boneid.report` writes to `reports/s2s_report.html`.

In [10]:
print(len(R.s2s_report_html(skeleton, kin, ground, idres, audit, truth)),
      "characters of report HTML (not written — run the module for that)")

440464 characters of report HTML (not written — run the module for that)
